# 05 - ECAPA-TDNN Experiments

Train and analyze the ECAPA-TDNN model with AAM-Softmax loss.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.evaluation.visualization import plot_training_curves, plot_confusion_matrix
from src.utils import count_parameters
from src.models.ecapa_tdnn import ECAPATDNN

plt.style.use('seaborn-v0_8-paper')

In [ ]:
# Load training history
history_path = Path('../results/ecapa_tdnn/training_history.json')
if history_path.exists():
    with open(history_path) as f:
        history = json.load(f)
    plot_training_curves(history, save_path='../results/ecapa_training_curves.png')
else:
    print('Training history not found. Run: python scripts/train_ecapa.py --config configs/ecapa_tdnn.yaml')

In [ ]:
# Model architecture summary
model = ECAPATDNN(num_speakers=10, channels=512, embedding_dim=192, scale=8)
print(f"Total trainable parameters: {count_parameters(model):,}")
print()
print(model)

In [ ]:
# Compare cross-entropy vs AAM-Softmax (if both histories exist)
ce_path = Path('../results/ecapa_tdnn_ce/training_history.json')
aam_path = Path('../results/ecapa_tdnn/training_history.json')

if ce_path.exists() and aam_path.exists():
    with open(ce_path) as f:
        ce_history = json.load(f)
    with open(aam_path) as f:
        aam_history = json.load(f)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(ce_history['val_acc'], label='Cross-Entropy', linewidth=2)
    ax1.plot(aam_history['val_acc'], label='AAM-Softmax', linewidth=2)
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Validation Accuracy')
    ax1.set_title('Loss Function Comparison')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(ce_history['val_loss'], label='Cross-Entropy', linewidth=2)
    ax2.plot(aam_history['val_loss'], label='AAM-Softmax', linewidth=2)
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Validation Loss')
    ax2.set_title('Validation Loss Comparison')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../results/loss_comparison.png', dpi=300)
    plt.show()
else:
    print('Need both CE and AAM-Softmax training histories for comparison.')

In [ ]:
# Load metrics
metrics_path = Path('../results/ecapa_tdnn/metrics.json')
if metrics_path.exists():
    with open(metrics_path) as f:
        metrics = json.load(f)
    print(f"Test Accuracy: {metrics['accuracy']*100:.1f}%")
    print(f"Top-5 Accuracy: {metrics.get('top5_accuracy', 0)*100:.1f}%")
    print(f"EER: {metrics.get('eer', 0)*100:.2f}%")
    
    if 'confusion_matrix' in metrics:
        cm = np.array(metrics['confusion_matrix'])
        plot_confusion_matrix(cm, save_path='../results/ecapa_confusion_matrix.png')
else:
    print('Metrics not found. Run evaluate_all.py first.')